In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

import pyarrow as pa
import pyarrow.parquet as pq

# ================== CONFIG ==================
DUMP = Path("metaspace_images_dump")

SAMPLES_PQ = DUMP / "msi_fm_samples5.parquet"   # sample list
MANIFEST_CAND_PQ = DUMP / "manifest_REBUILT2_with_molformer_candidates.parquet"

OUT_PQ = DUMP / "channels_with_candidates.parquet"

CAND_COLS = [
    "cand_names",
    "cand_smiles",
    "cand_inchi",
    "cand_pubchem_cids",
    "n_cand_molformer",
]

EXTRA_MANIFEST_COLS = [
    "sum_formula_clean", "adduct_clean", "peak_mz", "msm", "fdr"
]

CHUNK_ROWS = 200_000
PARQUET_COMPRESSION = "zstd"
# ============================================


# ---------- Robust path resolver ----------
def resolve_npz_path(sample_path_value, dump_root: Path) -> Path:
    """
    Handles:
      1) absolute paths
      2) paths that already include 'metaspace_images_dump/...'
      3) relative paths (either from CWD or from dump_root)
    Prevents double-prefixing dump_root.
    """
    sp = Path(str(sample_path_value))

    # 1) already exists as written
    if sp.exists():
        return sp

    # 2) sometimes stored relative to dump root, but with folders included
    #    e.g. "metaspace_images_dump/msi_fm_samples5/xxx.npz"
    #    If it contains dump_root name, strip everything up to it.
    parts = list(sp.parts)
    if dump_root.name in parts:
        j = parts.index(dump_root.name)
        sp2 = dump_root.parent.joinpath(*parts[j:])  # reconstruct from the dump folder
        if sp2.exists():
            return sp2

    # 3) try relative to dump_root (same tree)
    sp3 = dump_root / sp
    if sp3.exists():
        return sp3

    # 4) last resort: try dump_root / basename
    sp4 = dump_root / sp.name
    if sp4.exists():
        return sp4

    raise FileNotFoundError(f"Cannot locate NPZ: {sample_path_value}")


print("[LOAD] samples:", SAMPLES_PQ)
samples = pd.read_parquet(SAMPLES_PQ)

# deterministic order: dataset -> tile -> file
samples = samples.sort_values(
    by=["dataset_id", "tile_r", "tile_c", "tile_h", "tile_w", "sample_path"],
    kind="mergesort"
).reset_index(drop=True)

print("[LOAD] manifest candidates:", MANIFEST_CAND_PQ)
mani = pd.read_parquet(MANIFEST_CAND_PQ)

# Ensure manifest_row exists (must match what you stored in NPZ)
if "manifest_row" not in mani.columns:
    mani = mani.reset_index(drop=False).rename(columns={"index": "manifest_row"})

# Keep only needed columns
keep_cols = ["manifest_row"] + [c for c in (CAND_COLS + EXTRA_MANIFEST_COLS) if c in mani.columns]
missing_cands = [c for c in CAND_COLS if c not in mani.columns]
if missing_cands:
    print("[WARN] candidate columns missing in manifest parquet:", missing_cands)

mani_small = mani[keep_cols].copy()
mani_small["manifest_row"] = mani_small["manifest_row"].astype(np.int64)

# Index for fast join-by-id
mani_small = mani_small.set_index("manifest_row", drop=False)
print("[INFO] manifest rows:", len(mani_small))
print("[INFO] samples:", len(samples))

writer = None
buf = []
buf_n = 0

def flush():
    global writer, buf, buf_n
    if not buf:
        return
    df_chunk = pd.DataFrame(buf)
    table = pa.Table.from_pandas(df_chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_PQ, table.schema, compression=PARQUET_COMPRESSION)
    writer.write_table(table)
    buf = []
    buf_n = 0

print("[BUILD] writing channel-level parquet →", OUT_PQ)

for _, s in tqdm(samples.iterrows(), total=len(samples), dynamic_ncols=True):
    spath = resolve_npz_path(s["sample_path"], DUMP)

    z = np.load(spath, allow_pickle=False)
    mz = z["mz"].astype(np.float32, copy=False)
    mr = z["manifest_row"].astype(np.int64, copy=False)
    C = int(mr.shape[0])

    # Vectorized lookup for this sample’s manifest_rows (preserves channel order)
    vec_ok = True
    try:
        rows = mani_small.loc[mr]
    except KeyError:
        vec_ok = False
        rows = None

    for ci in range(C):
        rec = {
            "dataset_id": s["dataset_id"],
            "tile_r": int(s["tile_r"]),
            "tile_c": int(s["tile_c"]),
            "tile_h": int(s["tile_h"]),
            "tile_w": int(s["tile_w"]),
            "sample_path": str(s["sample_path"]),
            "npz_path_resolved": str(spath),

            "channels_in_sample": int(s["channels"]) if "channels" in s else int(C),

            "channel_idx": int(ci),
            "mz": float(mz[ci]) if np.isfinite(mz[ci]) else np.nan,
            "manifest_row": int(mr[ci]),
        }

        if vec_ok:
            r = rows.iloc[ci]
            for c in keep_cols:
                if c == "manifest_row":
                    continue
                rec[c] = r.get(c, None)
        else:
            mri = int(mr[ci])
            if mri in mani_small.index:
                r = mani_small.loc[mri]
                for c in keep_cols:
                    if c == "manifest_row":
                        continue
                    rec[c] = r.get(c, None)
            else:
                for c in keep_cols:
                    if c != "manifest_row":
                        rec[c] = None

        buf.append(rec)
        buf_n += 1
        if buf_n >= CHUNK_ROWS:
            flush()

flush()
if writer is not None:
    writer.close()

print("✅ DONE:", OUT_PQ)

# ---- Quick peek ----
df_peek = pd.read_parquet(OUT_PQ, columns=[
    "dataset_id","tile_r","tile_c","channel_idx","mz","manifest_row",
    *(c for c in CAND_COLS if c in keep_cols)
])
print("\n[PEEK]")
print(df_peek.head(10))

[LOAD] samples: metaspace_images_dump\msi_fm_samples5.parquet
[LOAD] manifest candidates: metaspace_images_dump\manifest_REBUILT2_with_molformer_candidates.parquet
[INFO] manifest rows: 695500
[INFO] samples: 5811
[BUILD] writing channel-level parquet → metaspace_images_dump\channels_with_candidates.parquet


100%|██████████| 5811/5811 [07:39<00:00, 12.66it/s]  


✅ DONE: metaspace_images_dump\channels_with_candidates.parquet

[PEEK]
             dataset_id  tile_r  tile_c  channel_idx          mz  \
0  2016-09-21_16h06m49s       0       0            0  283.264221   
1  2016-09-21_16h06m49s       0       0            1  255.232910   
2  2016-09-21_16h06m49s       0       0            2  153.019287   
3  2016-09-21_16h06m49s       0       0            3  124.007347   
4  2016-09-21_16h06m49s       0       0            4  227.201614   
5  2016-09-21_16h06m49s       0       0            5  199.170319   
6  2016-09-21_16h06m49s       0       0            6  241.217270   
7  2016-09-21_16h06m49s       0       0            7  599.320129   
8  2016-09-21_16h06m49s       0       0            8  286.943848   
9  2016-09-21_16h06m49s       0       0            9  219.050980   

   manifest_row                                         cand_names  \
0            54  Stearic acid; Ethyl hexadecanoate; 16-Methylhe...   
1            33  Butyl dodecanoate; Dode

In [1]:
import pandas as pd
df = pd.read_parquet("metaspace_images_dump/channels_with_candidates.parquet")
df

,dataset_id,tile_r,tile_c,tile_h,tile_w,sample_path,npz_path_resolved,channels_in_sample,channel_idx,mz,...,cand_names,cand_smiles,cand_inchi,cand_pubchem_cids,n_cand_molformer,sum_formula_clean,adduct_clean,peak_mz,msm,fdr
0,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,metaspace_images_dump\msi_fm_samples5\2016-09-...,32,0,283.264221,...,Stearic acid; Ethyl hexadecanoate; 16-Methylhe...,CCCCCCCCCCCCCCCCCC(=O)O ; CCCCCCCCCCCCCCCC(=O)...,InChI=1S/C18H36O2/c1-2-3-4-5-6-7-8-9-10-11-12-...,CID5281 ; CID12366 ; CID21859 ; CID94454,4,C18H36O2,-H,283.264215,0.992054,0.05
1,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,metaspace_images_dump\msi_fm_samples5\2016-09-...,32,1,255.232910,...,Butyl dodecanoate; Dodecyl butyrate; Hexyl dec...,CCCCCCCCCCCC(=O)OCCCC ; CCCCCCCCCCCCOC(=O)CCC ...,InChI=1S/C16H32O2/c1-3-5-7-8-9-10-11-12-13-14-...,CID61015 ; CID245572 ; CID82635 ; CID985 ; CID...,9,C16H32O2,-H,255.232915,0.990226,0.05
2,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,metaspace_images_dump\msi_fm_samples5\2016-09-...,32,2,153.019287,...,Protocatechuic acid; Gentisic acid; 2-Pyrocate...,C1=CC(=C(C=C1C(=O)O)O)O ; C1=CC(=C(C=C1O)C(=O)...,InChI=1S/C7H6O4/c8-5-2-1-4(7(10)11)3-6(5)9/h1-...,CID72 ; CID3469 ; CID19 ; CID9338 ; CID7424 ; ...,8,C7H6O4,-H,153.019293,0.988155,0.05
3,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,metaspace_images_dump\msi_fm_samples5\2016-09-...,32,3,124.007347,...,Taurine,C(CS(=O)(=O)O)N,"InChI=1S/C2H7NO3S/c3-1-2-7(4,5)6/h1-3H2,(H,4,5,6)",CID1123,1,C2H7NO3S,-H,124.007348,0.983255,0.05
4,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,metaspace_images_dump\msi_fm_samples5\2016-09-...,32,4,227.201614,...,"12-Methyltridecanoic acid; Hexanal octane-1,3-...",CC(C)CCCCCCCCCCC(=O)O ; CCCCCC1CCOC(O1)CCCCC ;...,InChI=1S/C14H28O2/c1-13(2)11-9-7-5-3-4-6-8-10-...,CID520298 ; CID526993 ; CID11005 ; CID3047764 ...,14,C14H28O2,-H,227.201614,0.981801,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164354,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,metaspace_images_dump\msi_fm_samples5\2026-01-...,8,3,441.079193,...,Kaempferol 3-O-arabinoside; Kaempferol 3-alpha...,C1[C@@H]([C@@H]([C@H]([C@@H](O1)OC2=C(OC3=CC(=...,InChI=1S/C20H18O10/c21-9-3-1-8(2-4-9)18-19(30-...,CID5481882 ; CID13245583 ; CID44258420 ; CID11...,4,C20H18O10,+Na,441.079196,0.931906,0.05
164355,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,metaspace_images_dump\msi_fm_samples5\2026-01-...,8,4,734.569397,...,PC(16:0/16:0); PC(14:0/18:0); PC(18:0/14:0); P...,CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)([O-])OCC[N...,InChI=1S/C40H80NO8P/c1-6-8-10-12-14-16-18-20-2...,CID452110 ; CID131150 ; CID3082163 ; CID446872...,11,C40H80NO8P,+H,734.569410,0.921602,0.05
164356,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,metaspace_images_dump\msi_fm_samples5\2026-01-...,8,5,837.681946,...,SM(d18:0/24:1(15Z)),CCCCCCCCCCCCCCC[C@H]([C@H](COP(=O)([O-])OCC[N+...,InChI=1S/C47H95N2O6P/c1-6-8-10-12-14-16-18-20-...,CID44260133,1,C47H95N2O6P,+Na,837.681974,0.899866,0.05
164357,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,metaspace_images_dump\msi_fm_samples5\2026-01-...,8,6,673.589355,...,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...,CCCCCCCC/C=C\CCCCCCCC(=O)O[C@H]1CC[C@@]2([C@H]...,InChI=1S/C45H78O2/c1-7-8-9-10-11-12-13-14-15-1...,CID5283632 ; CID53477793,2,C45H78O2,+Na,673.589381,0.867454,0.05


In [3]:
df.columns

Index(['dataset_id', 'tile_r', 'tile_c', 'tile_h', 'tile_w', 'sample_path',
       'npz_path_resolved', 'channels_in_sample', 'channel_idx', 'mz',
       'manifest_row', 'cand_names', 'cand_smiles', 'cand_inchi',
       'cand_pubchem_cids', 'n_cand_molformer', 'sum_formula_clean',
       'adduct_clean', 'peak_mz', 'msm', 'fdr'],
      dtype='object')

In [7]:
df.iloc[0]["npz_path_resolved"]

'metaspace_images_dump\\msi_fm_samples5\\2016-09-21_16h06m49s_r0_c0_C32.npz'

In [8]:
df.iloc[0]["sample_path"]

'metaspace_images_dump\\msi_fm_samples5\\2016-09-21_16h06m49s_r0_c0_C32.npz'

In [1]:
# ============================================================
# (3) Build LONG exploded table:
#   channel_candidate_pairs.parquet
#
# One row per (channel_row, candidate_molecule).
#
# Inputs:
#   - metaspace_images_dump/channels_with_candidates.parquet
#   - metaspace_images_dump/manifest_cand_molformer_rows_flat.npy
#   - metaspace_images_dump/manifest_cand_molformer_offsets.npy
# Optional:
#   - metaspace_images_dump/molformer_pubchem_index_enriched_semantic.parquet
#     (joins molecule metadata like cid/name/smiles/inchikey/text_semantic)
#
# Output:
#   - metaspace_images_dump/channel_candidate_pairs.parquet
#
# Notes:
#   - Chunked Parquet writing (memory-safe).
#   - Keeps channel order identity: dataset/tile/channel_idx + mz + manifest_row.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

# ================== CONFIG ==================
DUMP = Path("metaspace_images_dump")

CHAN_PQ = DUMP / "channels_with_candidates.parquet"
FLAT    = DUMP / "manifest_cand_molformer_rows_flat.npy"
OFFS    = DUMP / "manifest_cand_molformer_offsets.npy"

# Optional metadata join (recommended)
MOL_INDEX_PQ = DUMP / "molformer_pubchem_index_enriched_semantic.parquet"
JOIN_MOL_METADATA = True

OUT_PQ = DUMP / "channel_candidate_pairs.parquet"

# write chunk size (rows written per flush)
CHUNK_OUT_ROWS = 500_000
PARQUET_COMPRESSION = "zstd"

# Choose which channel columns to carry into long table
CHAN_KEEP_COLS = [
    "dataset_id", "tile_r", "tile_c", "tile_h", "tile_w",
    "sample_path", "channel_idx", "sum_formula_clean", "adduct_clean", "mz", "manifest_row"
]

# Choose which molecule metadata columns to bring (if file exists)
MOL_META_COLS = [
    "cid", "name", "smiles", "inchi", "inchikey", "chebi_synonyms_topk",
    "chebi_accession", "hmdb_id",
    "hmdb_taxonomy",
    "pathbank_pathway_names", "pathbank_pathway_desc",   
    "text_semantic"
]
# ============================================

print("[LOAD] channels:", CHAN_PQ)
ch = pd.read_parquet(CHAN_PQ)

missing = [c for c in CHAN_KEEP_COLS if c not in ch.columns]
if missing:
    raise RuntimeError(f"channels_with_candidates missing required cols: {missing}")

# Keep only needed cols to reduce RAM
ch = ch[CHAN_KEEP_COLS].copy()

print("[LOAD] manifest cand CSR arrays")
flat = np.load(FLAT, mmap_mode="r")
offs = np.load(OFFS, mmap_mode="r")

# Optional molecule index metadata
mol_df = None
if JOIN_MOL_METADATA and MOL_INDEX_PQ.exists():
    print("[LOAD] molecule index:", MOL_INDEX_PQ)
    mol_df = pd.read_parquet(MOL_INDEX_PQ)
    # Ensure we can join by molformer_row (row index in embeddings)
    mol_df = mol_df.reset_index(drop=False).rename(columns={"index": "molformer_row"})
    keep_m = ["molformer_row"] + [c for c in MOL_META_COLS if c in mol_df.columns]
    mol_df = mol_df[keep_m].copy()
    print("[INFO] mol meta cols:", [c for c in keep_m if c != "molformer_row"])
else:
    if JOIN_MOL_METADATA:
        print("[WARN] molecule index not found, writing pairs without metadata:", MOL_INDEX_PQ)
    mol_df = None

print("[BUILD] exploding channel → candidate molecules →", OUT_PQ)

writer = None
buf = []
buf_n = 0

def flush():
    global writer, buf, buf_n
    if not buf:
        return
    df = pd.DataFrame(buf)
    tab = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_PQ, tab.schema, compression=PARQUET_COMPRESSION)
    writer.write_table(tab)
    buf = []
    buf_n = 0

# Iterate channels in-order; channel_row is stable index into channels parquet
mrs = ch["manifest_row"].astype(np.int64).to_numpy()

for i in tqdm(range(len(ch)), desc="Explode", dynamic_ncols=True):
    mr = int(mrs[i])
    a = int(offs[mr])
    b = int(offs[mr + 1])
    cand = flat[a:b]
    if cand.size == 0:
        continue

    row = ch.iloc[i]
    # Emit one record per candidate
    for r in cand:
        buf.append({
            "channel_row": int(i),                 # index into channels_with_candidates.parquet
            "dataset_id": row["dataset_id"],
            "tile_r": int(row["tile_r"]),
            "tile_c": int(row["tile_c"]),
            "tile_h": int(row["tile_h"]),
            "tile_w": int(row["tile_w"]),
            "sample_path": row["sample_path"],
            "channel_idx": int(row["channel_idx"]),
            "mz": float(row["mz"]) if pd.notna(row["mz"]) else np.nan,
            "manifest_row": mr,
            "molformer_row": int(r),               # row id into molformer_pubchem_embeddings.npy
        })
        buf_n += 1
        if buf_n >= CHUNK_OUT_ROWS:
            flush()

flush()
if writer is not None:
    writer.close()

print("✅ wrote:", OUT_PQ)

# ---------- Optional: join molecule metadata into a NEW parquet ----------
if mol_df is not None:
    OUT_JOINED = DUMP / "channel_candidate_pairs_with_molmeta.parquet"
    print("[JOIN] adding molecule metadata →", OUT_JOINED)

    pairs = pd.read_parquet(OUT_PQ)
    pairs2 = pairs.merge(mol_df, on="molformer_row", how="left")
    pairs2.to_parquet(OUT_JOINED, index=False, engine="pyarrow")
    print("✅ wrote:", OUT_JOINED)

    # quick peek
    print("\n[PEEK joined]")
    print(pairs2.head(10))
else:
    # quick peek
    print("\n[PEEK]")
    print(pd.read_parquet(OUT_PQ).head(10))


[LOAD] channels: metaspace_images_dump\channels_with_candidates.parquet
[LOAD] manifest cand CSR arrays
[LOAD] molecule index: metaspace_images_dump\molformer_pubchem_index_enriched_semantic.parquet
[INFO] mol meta cols: ['cid', 'name', 'smiles', 'inchi', 'inchikey', 'chebi_synonyms_topk', 'chebi_accession', 'hmdb_id', 'hmdb_taxonomy', 'pathbank_pathway_names', 'pathbank_pathway_desc', 'text_semantic']
[BUILD] exploding channel → candidate molecules → metaspace_images_dump\channel_candidate_pairs.parquet


Explode: 100%|██████████| 164359/164359 [00:55<00:00, 2961.01it/s]


✅ wrote: metaspace_images_dump\channel_candidate_pairs.parquet
[JOIN] adding molecule metadata → metaspace_images_dump\channel_candidate_pairs_with_molmeta.parquet
✅ wrote: metaspace_images_dump\channel_candidate_pairs_with_molmeta.parquet

[PEEK joined]
   channel_row            dataset_id  tile_r  tile_c  tile_h  tile_w  \
0            0  2016-09-21_16h06m49s       0       0     189     268   
1            0  2016-09-21_16h06m49s       0       0     189     268   
2            0  2016-09-21_16h06m49s       0       0     189     268   
3            0  2016-09-21_16h06m49s       0       0     189     268   
4            1  2016-09-21_16h06m49s       0       0     189     268   
5            1  2016-09-21_16h06m49s       0       0     189     268   
6            1  2016-09-21_16h06m49s       0       0     189     268   
7            1  2016-09-21_16h06m49s       0       0     189     268   
8            1  2016-09-21_16h06m49s       0       0     189     268   
9            1  2016-09-2

In [9]:
import pandas as pd
df = pd.read_parquet("metaspace_images_dump/channel_candidate_pairs.parquet")
df

,channel_row,dataset_id,tile_r,tile_c,tile_h,tile_w,sample_path,channel_idx,mz,manifest_row,molformer_row
0,0,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,0,283.264221,54,23685
1,0,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,0,283.264221,54,12770
2,0,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,0,283.264221,54,1783
3,0,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,0,283.264221,54,13846
4,1,2016-09-21_16h06m49s,0,0,189,268,metaspace_images_dump\msi_fm_samples5\2016-09-...,1,255.232910,33,6918
...,...,...,...,...,...,...,...,...,...,...,...
1756780,164358,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,7,721.477844,695485,16922
1756781,164358,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,7,721.477844,695485,16369
1756782,164358,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,7,721.477844,695485,16387
1756783,164358,2026-01-18_22h19m43s,0,0,138,108,metaspace_images_dump\msi_fm_samples5\2026-01-...,7,721.477844,695485,16679


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

# ================== CONFIG ==================
DUMP = Path("metaspace_images_dump")

CHAN_PQ = DUMP / "channels_with_candidates.parquet"

# These are the MANIFEST-level CSR arrays you already built (spectral)
MANIFEST_FLAT = DUMP / "manifest_cand_molformer_rows_flat.npy"
MANIFEST_OFFS = DUMP / "manifest_cand_molformer_offsets.npy"

# Embeddings (same global molecule row order)
Z_MOL_NPY = DUMP / "molformer_pubchem_embeddings.npy"
Z_TXT_NPY = DUMP / "cand_molecule_biolinkbert_embeddings.npy"

# Outputs (channel-level CSR)
OUT_FLAT = DUMP / "channel_cand_rows_flat.npy"
OUT_OFFS = DUMP / "channel_cand_rows_offsets.npy"
# ============================================

print("[LOAD] channels:", CHAN_PQ)
ch = pd.read_parquet(CHAN_PQ, columns=["manifest_row"])
mr = ch["manifest_row"].astype(np.int64).to_numpy()

print("[LOAD] manifest CSR arrays")
mflat = np.load(MANIFEST_FLAT, mmap_mode="r")
moffs = np.load(MANIFEST_OFFS, mmap_mode="r")

# Sanity: embeddings share the same row space
Zmol = np.load(Z_MOL_NPY, mmap_mode="r")
Ztxt = np.load(Z_TXT_NPY, mmap_mode="r")
if Zmol.shape[0] != Ztxt.shape[0]:
    raise RuntimeError(f"MolFormer rows ({Zmol.shape[0]}) != BioLinkBERT rows ({Ztxt.shape[0]})")
n_mols = Zmol.shape[0]
print("[INFO] global molecule rows:", n_mols)

print("[PASS1] counting total channel-candidate links...")
counts = np.zeros(len(mr), dtype=np.int32)
total = 0
for i, mri in enumerate(tqdm(mr, total=len(mr), dynamic_ncols=True)):
    mri = int(mri)
    a = int(moffs[mri]); b = int(moffs[mri + 1])
    # filter to valid rows (defensive)
    c = 0
    if b > a:
        cand = mflat[a:b]
        # keep only in-range ids
        c = int(np.sum((cand >= 0) & (cand < n_mols)))
    counts[i] = c
    total += c

print(f"[INFO] channels: {len(mr):,}")
print(f"[INFO] total channel→candidate links: {total:,}")
print(f"[INFO] channels with zero candidates: {(counts==0).sum():,} ({(counts==0).mean()*100:.2f}%)")

print("[PASS2] building channel CSR flat+offsets...")
offs = np.zeros(len(mr) + 1, dtype=np.int64)
np.cumsum(counts, out=offs[1:])  # offs[0]=0, offs[i+1]=sum_{0..i} counts

flat = np.empty(total, dtype=np.int32)

cursor = 0
for i, mri in enumerate(tqdm(mr, total=len(mr), dynamic_ncols=True)):
    mri = int(mri)
    a = int(moffs[mri]); b = int(moffs[mri + 1])
    if b <= a:
        continue
    cand = mflat[a:b].astype(np.int32, copy=False)
    # keep only valid ids
    cand = cand[(cand >= 0) & (cand < n_mols)]
    if cand.size == 0:
        continue
    flat[cursor:cursor + cand.size] = cand
    cursor += cand.size

if cursor != total:
    # Should not happen unless something weird in filtering; trim if needed
    print(f"[WARN] cursor {cursor} != total {total}; trimming outputs")
    flat = flat[:cursor]
    offs = offs.copy()
    # recompute offs from actual written counts
    # easiest: rebuild counts from offs deltas won't match; just rebuild offs by scanning again
    counts2 = np.zeros(len(mr), dtype=np.int64)
    cursor2 = 0
    for i, mri in enumerate(mr):
        a = int(moffs[int(mri)]); b = int(moffs[int(mri) + 1])
        if b > a:
            cand = mflat[a:b]
            counts2[i] = int(np.sum((cand >= 0) & (cand < n_mols)))
        cursor2 += counts2[i]
        offs[i+1] = cursor2
    total = cursor

print("[SAVE] writing channel CSR arrays")
np.save(OUT_FLAT, flat)
np.save(OUT_OFFS, offs)
print("✅ saved:", OUT_FLAT, flat.shape, flat.dtype)
print("✅ saved:", OUT_OFFS, offs.shape, offs.dtype)


[LOAD] channels: metaspace_images_dump\channels_with_candidates.parquet
[LOAD] manifest CSR arrays
[INFO] global molecule rows: 30478
[PASS1] counting total channel-candidate links...


100%|██████████| 164359/164359 [00:01<00:00, 99436.94it/s] 


[INFO] channels: 164,359
[INFO] total channel→candidate links: 1,756,785
[INFO] channels with zero candidates: 6,512 (3.96%)
[PASS2] building channel CSR flat+offsets...


100%|██████████| 164359/164359 [00:01<00:00, 112055.54it/s]


[SAVE] writing channel CSR arrays
✅ saved: metaspace_images_dump\channel_cand_rows_flat.npy (1756785,) int32
✅ saved: metaspace_images_dump\channel_cand_rows_offsets.npy (164360,) int64


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

DUMP = Path("metaspace_images_dump")

ch = pd.read_parquet(DUMP/"channels_with_candidates.parquet")
flat = np.load(DUMP/"channel_cand_rows_flat.npy", mmap_mode="r")
offs = np.load(DUMP/"channel_cand_rows_offsets.npy", mmap_mode="r")

Zmol = np.load(DUMP/"molformer_pubchem_embeddings.npy", mmap_mode="r")
Ztxt = np.load(DUMP/"cand_molecule_biolinkbert_embeddings.npy", mmap_mode="r")

channel_row = 12345

a, b = int(offs[channel_row]), int(offs[channel_row+1])
mol_rows = flat[a:b]                 # all candidate molecule row IDs
E_mol = Zmol[mol_rows]               # (n_cand, 768)
E_txt = Ztxt[mol_rows]               # (n_cand, 768)

print("channel mz:", ch.loc[channel_row, "mz"])
print("n candidates:", len(mol_rows))
print("mol emb shape:", E_mol.shape, "txt emb shape:", E_txt.shape)


channel mz: 861.5498046875
n candidates: 8
mol emb shape: (8, 768) txt emb shape: (8, 768)
